# Train a HEST transformer from saved tensors
Run verification or training independently. Edit tensor paths in each cell.


## 8a. Verify saved tensors
Edit four .pt paths in the cell. Shape checks cannot establish patch/gene identity or order.


In [ ]:
from pathlib import Path
import sys
_here = Path.cwd().resolve()
PROJECT_ROOT = next((p for p in (_here, *_here.parents) if (p / 'scripts').is_dir()), None)
if PROJECT_ROOT is None:
    raise RuntimeError("Start Jupyter from HEST_Local or its notebook folder.")
if str(PROJECT_ROOT / 'scripts') not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / 'scripts'))

# Edit these paths to point to your tensors.
TRAIN_X = PROJECT_ROOT / 'training_run' / 'X_train_pca250.pt'
TRAIN_Y = PROJECT_ROOT / 'training_run' / 'y_train.pt'
TEST_X = PROJECT_ROOT / 'training_run' / 'X_test_pca250.pt'
TEST_Y = PROJECT_ROOT / 'training_run' / 'y_test.pt'
import importlib, pandas as pd
import hest_transformer_model as modeling
importlib.reload(modeling)
report = modeling.verify_training_tensors(TRAIN_X, TRAIN_Y, TEST_X, TEST_Y)
display(pd.DataFrame(report['files']))
print('Shape verification passed. Ensure matching patch and gene order.')


## Train from four tensor files
Edit the four tensor paths and parameters in the following cell. Verification is optional.


In [ ]:
from pathlib import Path
import sys
_here = Path.cwd().resolve()
PROJECT_ROOT = next((p for p in (_here, *_here.parents) if (p / 'scripts').is_dir()), None)
if PROJECT_ROOT is None:
    raise RuntimeError("Start Jupyter from HEST_Local or its notebook folder.")
if str(PROJECT_ROOT / 'scripts') not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / 'scripts'))

# Edit tensor paths and training parameters directly.
TRAIN_X = PROJECT_ROOT / 'training_run' / 'X_train_pca250.pt'
TRAIN_Y = PROJECT_ROOT / 'training_run' / 'y_train.pt'
TEST_X = PROJECT_ROOT / 'training_run' / 'X_test_pca250.pt'
TEST_Y = PROJECT_ROOT / 'training_run' / 'y_test.pt'
OUTPUT_DIR = PROJECT_ROOT / 'training_run' / 'model'
EPOCHS = 30
BATCH_SIZE = 256
LEARNING_RATE = 1e-3
D_MODEL = 64
N_HEADS = 4
N_LAYERS = 2
DROPOUT = 0.1
SEED = 42
DEVICE = None  # None automatically uses CUDA if available.
import importlib
import hest_transformer_model as modeling
importlib.reload(modeling)
model, metrics = modeling.train_transformer(
    TRAIN_X, TRAIN_Y, TEST_X, TEST_Y, output_dir=OUTPUT_DIR,
    epochs=EPOCHS, batch_size=BATCH_SIZE, learning_rate=LEARNING_RATE,
    d_model=D_MODEL, n_heads=N_HEADS, n_layers=N_LAYERS,
    dropout=DROPOUT, seed=SEED, device=DEVICE)
print('Final training MSE:', metrics['train_loss'])
print('Final test MSE:', metrics['test_loss'])
print('Mean test PCC:', metrics['mean_pcc'])
print('Mean std-normalized MAE:', metrics['mean_std_normalized_mae'])
print('Outputs:', OUTPUT_DIR)
